# 01 — Data Collection

**Goal:** Pull MoneyPuck CSVs (all situations + PP + SH splits) for seasons 2019–2024,
join with salary data from the SQLite database, apply the GP ≥ 40 quality filter,
and export a raw combined CSV for the next notebook to clean.

**Stop point:** Review the shape, completeness, and any obvious data quality issues
before moving to feature engineering.

---
### Decisions already locked in
- Seasons: 2019–2024 (2025 = current, excluded from historical training pairs)
- Situations pulled: `all`, `5on5`, `pp`, `sh`
- GP filter: rows with < 40 games played are excluded from training data
- Salary pairing: each stat season is paired with the salary active at contract signing,
  estimated via `expiry_season - contract_years`
- Position groups: F (C/LW/RW combined), D, G — separate models
- Clauses: will be binary-encoded in notebook 02

In [ ]:
import sys, os
import sqlite3
import json
import requests
import io
import csv
import pandas as pd
import numpy as np

# ── Path to the backend SQLite database ──────────────────────────────────────
DB_PATH = os.path.join(
    os.path.dirname(os.getcwd()), "backend", "salary_cache.db"
)
print(f"DB path: {DB_PATH}")
print(f"DB exists: {os.path.exists(DB_PATH)}")

## 1. Load Salary Data from SQLite

In [ ]:
conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row

salary_df = pd.read_sql_query("""
    SELECT
        player_id,
        player_name,
        team,
        position,
        aav,
        contract_years,
        expiry_season,
        fa_type,
        clauses,
        source
    FROM salaries
    WHERE aav IS NOT NULL AND aav > 0
""", conn)

conn.close()

print(f"Salary rows: {len(salary_df):,}")
print(f"Sources: {salary_df['source'].value_counts().to_dict()}")
salary_df.head()

In [ ]:
# Estimate the season the contract was SIGNED
# Logic: signing_year = expiry_season - contract_years
# If we can't compute it, default to current season (2024)
def estimate_signing_year(row):
    try:
        if pd.notna(row['expiry_season']) and pd.notna(row['contract_years']) and int(row['contract_years']) > 0:
            return int(row['expiry_season']) - int(row['contract_years'])
    except (ValueError, TypeError):
        pass
    return 2024  # default to most recent complete season

salary_df['signing_year'] = salary_df.apply(estimate_signing_year, axis=1)

print("Signing year distribution:")
print(salary_df['signing_year'].value_counts().sort_index())

## 2. Fetch MoneyPuck CSVs

We pull four situations per season:
- `all` — full-game baseline stats
- `5on5` — even-strength only (removes PP inflation)
- `pp` — power play (isolates PP specialists)
- `sh` — shorthanded (deployment indicator)

Columns pulled per situation are documented below. Each situation's columns
will be prefixed (e.g. `pp_goals`, `5on5_xgf_pct`) before merging.

In [ ]:
MONEYPUCK_BASE = "https://moneypuck.com/moneypuck/playerData/seasonSummary"
SEASONS = [str(y) for y in range(2019, 2025)]  # 2019–2024
SITUATIONS = ["all", "5on5", "pp", "sh"]

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
}

# Columns we want from each situation (raw MoneyPuck names)
ALL_COLS = [
    "playerId", "season", "name", "team", "position", "situation",
    "games_played", "icetime",
    "gameScore",
    "onIce_xGoalsPercentage", "offIce_xGoalsPercentage",
    "onIce_corsiPercentage",  "offIce_corsiPercentage",
    "I_F_goals", "I_F_primaryAssists", "I_F_secondaryAssists",
    "I_F_xGoals", "I_F_highDangerGoals", "I_F_highDangerShots",
    "I_F_shotsOnGoal", "I_F_rebounds",
    "I_F_hits", "I_F_takeaways", "I_F_giveaways", "I_F_dZoneGiveaways",
    "I_F_penalityMinutes", "penaltiesDrawn",
    "penalties",
    "shotsBlockedByPlayer",
    "faceoffsWon", "faceoffsLost",
    "OnIce_F_xGoals", "OnIce_A_xGoals",
    "OnIce_F_shotAttempts", "OnIce_A_shotAttempts",
    "OnIce_F_highDangerShots", "OnIce_A_highDangerShots",
    "I_F_oZoneShiftStarts", "I_F_dZoneShiftStarts", "I_F_neutralZoneShiftStarts",
]

# For PP and SH we only need icetime + production
SPLIT_COLS = [
    "playerId", "season", "situation",
    "icetime",
    "I_F_goals", "I_F_primaryAssists", "I_F_secondaryAssists",
    "I_F_xGoals", "I_F_highDangerGoals",
    "OnIce_F_xGoals", "OnIce_A_xGoals",
    "OnIce_F_shotAttempts", "OnIce_A_shotAttempts",
]

print("Column sets defined.")
print(f"  all/5on5: {len(ALL_COLS)} columns")
print(f"  pp/sh:    {len(SPLIT_COLS)} columns")

In [ ]:
def fetch_mp_csv(season, player_type, situation):
    """Fetch one MoneyPuck CSV and return a DataFrame filtered to one situation."""
    url = f"{MONEYPUCK_BASE}/{season}/regular/{player_type}.csv"
    try:
        resp = requests.get(url, headers=HEADERS, timeout=30)
        resp.raise_for_status()
        df = pd.read_csv(io.StringIO(resp.text), low_memory=False)
        df = df[df["situation"] == situation].copy()
        df["season"] = season  # ensure string season
        return df
    except Exception as e:
        print(f"  WARN: {url} failed — {e}")
        return pd.DataFrame()


def safe_cols(df, wanted):
    """Return only the columns that actually exist in df."""
    return [c for c in wanted if c in df.columns]


print("Fetch helpers defined.")

In [ ]:
# ── Pull all skater data ─────────────────────────────────────────────────────
# This cell takes ~60–90 seconds (6 seasons × 4 situations × 1 HTTP request)

frames_all   = []  # situation='all'
frames_5on5  = []  # situation='5on5'
frames_pp    = []  # situation='pp'
frames_sh    = []  # situation='sh'

for season in SEASONS:
    print(f"Season {season}...", end=" ")

    df_all = fetch_mp_csv(season, "skaters", "all")
    if not df_all.empty:
        frames_all.append(df_all[safe_cols(df_all, ALL_COLS)])

    df_5on5 = fetch_mp_csv(season, "skaters", "5on5")
    if not df_5on5.empty:
        frames_5on5.append(df_5on5[safe_cols(df_5on5, ALL_COLS)])

    df_pp = fetch_mp_csv(season, "skaters", "pp")
    if not df_pp.empty:
        frames_pp.append(df_pp[safe_cols(df_pp, SPLIT_COLS)])

    df_sh = fetch_mp_csv(season, "skaters", "sh")
    if not df_sh.empty:
        frames_sh.append(df_sh[safe_cols(df_sh, SPLIT_COLS)])

    print(f"all={len(df_all)}, 5on5={len(df_5on5)}, pp={len(df_pp)}, sh={len(df_sh)}")

skaters_all  = pd.concat(frames_all,  ignore_index=True) if frames_all  else pd.DataFrame()
skaters_5on5 = pd.concat(frames_5on5, ignore_index=True) if frames_5on5 else pd.DataFrame()
skaters_pp   = pd.concat(frames_pp,   ignore_index=True) if frames_pp   else pd.DataFrame()
skaters_sh   = pd.concat(frames_sh,   ignore_index=True) if frames_sh   else pd.DataFrame()

print(f"\nTotal rows — all: {len(skaters_all)}, 5on5: {len(skaters_5on5)}, pp: {len(skaters_pp)}, sh: {len(skaters_sh)}")

## 3. Merge Situations into One Row per Player-Season

Each player-season gets one row with:
- All-situations columns (unprefixed) as the baseline
- 5-on-5 columns prefixed `ev_` (even-strength)
- PP columns prefixed `pp_`
- SH columns prefixed `sh_`

In [ ]:
def prefix_cols(df, prefix, keep_keys=("playerId", "season")):
    """Rename all columns except the join keys with a prefix."""
    rename = {c: f"{prefix}_{c}" for c in df.columns if c not in keep_keys + ("situation",)}
    return df.drop(columns=["situation"], errors="ignore").rename(columns=rename)

skaters_5on5_p = prefix_cols(skaters_5on5, "ev")
skaters_pp_p   = prefix_cols(skaters_pp,   "pp")
skaters_sh_p   = prefix_cols(skaters_sh,   "sh")

# Start from 'all' as the base, merge each split on playerId + season
merged = skaters_all.copy()
merged = merged.drop(columns=["situation"], errors="ignore")

for split_df, label in [
    (skaters_5on5_p, "5on5"),
    (skaters_pp_p,   "pp"),
    (skaters_sh_p,   "sh"),
]:
    if not split_df.empty:
        merged = merged.merge(
            split_df, on=["playerId", "season"], how="left", suffixes=("", f"_dup_{label}")
        )
        # Drop any accidental duplicate columns
        dup_cols = [c for c in merged.columns if f"_dup_{label}" in c]
        merged = merged.drop(columns=dup_cols)

print(f"Merged shape: {merged.shape}")
print(f"Columns: {list(merged.columns)}")

## 4. Apply GP ≥ 40 Filter

In [ ]:
before = len(merged)
merged = merged[merged["games_played"] >= 40].copy()
after  = len(merged)

print(f"Rows before GP filter: {before:,}")
print(f"Rows after  GP filter: {after:,}  (removed {before - after:,} low-GP seasons)")
print()
print("Games played distribution (post-filter):")
print(merged["games_played"].describe().round(1))

## 5. Join with Salary Data

**Key logic:** We pair each player's stats from season `S` with the salary
they were earning in season `S` — i.e., we only use salary records where the
contract was *active* during season `S` (signing_year ≤ S < expiry_season).

This avoids the bug of pairing a 2019 stat line with a 2025 contract extension.

In [ ]:
merged["playerId"] = merged["playerId"].astype(str)
merged["season_int"] = merged["season"].astype(int)

salary_df["player_id"] = salary_df["player_id"].astype(str)

# Cross join stats × salaries, keep only rows where contract was active that season
combined = merged.merge(
    salary_df,
    left_on="playerId",
    right_on="player_id",
    how="inner",
)

# Filter: stat season must fall within the contract window
# signing_year <= stat_season AND (stat_season < expiry_season OR expiry_season is null)
mask = (
    (combined["signing_year"] <= combined["season_int"]) &
    (
        combined["expiry_season"].isna() |
        (combined["season_int"] < combined["expiry_season"])
    )
)
combined = combined[mask].copy()

print(f"Combined rows (stats matched to active contract): {len(combined):,}")
print(f"Unique players: {combined['playerId'].nunique():,}")
print(f"Seasons covered: {sorted(combined['season_int'].unique())}")

In [ ]:
# If a player has multiple salary records matched (rare), keep the best-matched one
# (the contract whose signing_year is closest to the stat season)
combined["signing_dist"] = combined["season_int"] - combined["signing_year"]
combined = (
    combined
    .sort_values("signing_dist")
    .groupby(["playerId", "season_int"], as_index=False)
    .first()
)

print(f"After dedup: {len(combined):,} rows")
print(f"\nAAV distribution:")
print(combined["aav"].describe().apply(lambda x: f"${x:,.0f}"))

## 6. Position Breakdown

Verify we have sufficient training data per position group before proceeding.

In [ ]:
POS_GROUPS = {
    "C": "F", "L": "F", "LW": "F", "R": "F", "RW": "F", "W": "F", "F": "F",
    "D": "D", "LD": "D", "RD": "D",
    "G": "G",
}

combined["pos_group"] = combined["position_x"].str.upper().map(POS_GROUPS).fillna("F")

print("Rows per position group:")
print(combined["pos_group"].value_counts())
print()
print("Unique players per group:")
print(combined.groupby("pos_group")["playerId"].nunique())

## 7. Missing Data Audit

Before exporting, check which columns have significant nulls.
We need to decide: impute with median, impute with 0, or drop the feature.

In [ ]:
# Show null rates for key columns only
key_cols = [
    "games_played", "icetime", "gameScore",
    "onIce_xGoalsPercentage", "offIce_xGoalsPercentage", "onIce_corsiPercentage",
    "I_F_goals", "I_F_xGoals", "I_F_highDangerGoals",
    "I_F_hits", "I_F_takeaways", "I_F_giveaways", "I_F_dZoneGiveaways",
    "faceoffsWon", "faceoffsLost",
    "shotsBlockedByPlayer",
    "OnIce_F_xGoals", "OnIce_A_xGoals",
    "OnIce_F_shotAttempts", "OnIce_A_shotAttempts",
    "OnIce_F_highDangerShots", "OnIce_A_highDangerShots",
    "pp_icetime", "pp_I_F_goals", "pp_I_F_xGoals",
    "sh_icetime",
    "aav", "fa_type", "clauses",
]
existing = [c for c in key_cols if c in combined.columns]
null_summary = combined[existing].isnull().mean().mul(100).round(1)
null_summary = null_summary[null_summary > 0].sort_values(ascending=False)

print("Columns with missing values (% null):")
print(null_summary.to_string())

## 8. Sample Rows — Spot Check

Visually verify the data looks correct for a few known players before exporting.

In [ ]:
spot_check_cols = [
    "name", "season_int", "pos_group", "games_played",
    "I_F_goals", "I_F_primaryAssists",
    "onIce_xGoalsPercentage", "offIce_xGoalsPercentage",
    "aav", "fa_type", "clauses", "signing_year",
]
existing_spot = [c for c in spot_check_cols if c in combined.columns]

# Show a few well-known players for sanity check
well_known = ["Connor McDavid", "Nathan MacKinnon", "Auston Matthews", "Cale Makar", "Sidney Crosby"]
spot = combined[combined["name"].isin(well_known)][existing_spot].sort_values(["name", "season_int"])
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
spot

## 9. Export Raw Combined Dataset

Save to `data/raw_combined.csv`. This is the **unprocessed** merge — no feature
engineering, no scaling, no imputation. Notebook 02 will handle all of that.

In [ ]:
out_dir = os.path.join(os.path.dirname(os.getcwd()), "notebooks", "data")
os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir, "raw_combined.csv")

combined.to_csv(out_path, index=False)

print(f"Saved {len(combined):,} rows × {len(combined.columns)} columns")
print(f"→ {out_path}")
print()
print("=" * 60)
print("STOP POINT — review the output above before running notebook 02")
print("Key things to check:")
print("  1. Position group row counts — enough data for D and G models?")
print("  2. Null rates — any key features mostly missing?")
print("  3. Spot check — do known players look right?")
print("  4. AAV distribution — any obvious outliers (entry-level, LTIR)?")
print("=" * 60)

---
## Next: Notebook 02 — Feature Engineering

Once you've reviewed the output above, open `02_feature_engineering.ipynb`.
That notebook will:
- Compute per-60 rate stats from the raw counting columns
- Binary-encode clauses (NMC, NTC, modified NTC)
- Cap-normalize AAV targets by signing-year cap ceiling
- Build position-specific feature matrices (F, D, G)
- Handle remaining nulls (imputation strategy per column type)
- Export `data/features_F.csv`, `data/features_D.csv`, `data/features_G.csv`